In [1]:
%pip install geopandas -q


StatementMeta(, f45c4e44-fdc4-4b8a-9204-a08a4f607000, 7, Finished, Available, Finished, True)


Note: you may need to restart the kernel to use updated packages.



In [4]:
import geopandas as gpd, pandas as pd, numpy as np, os, re, unicodedata
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
from scipy.spatial import cKDTree
from pyproj import Transformer

# ---------- Paramètres ----------
BRONZE = "/lakehouse/default/Files"
notebookutils.fs.mount("abfss://citiz_hackathon@onelake.dfs.fabric.microsoft.com/lakehouse_silver.Lakehouse", "/silver")
S = notebookutils.fs.getMountPath("/silver") + "/Files"
os.makedirs(f"{S}/acces_5min", exist_ok=True)
MARCHE = 400          # 5 min à pied = 400 m par les rues

carreaux = gpd.read_parquet(f"{S}/carreaux/carreaux_toulon_marseille.parquet").to_crs(2154).reset_index(drop=True)
arrets   = gpd.read_parquet(f"{S}/arrets/arrets_toulon_marseille.parquet").to_crs(2154)

# ---------- 1. Regrouper les quais d'un même arrêt = 1 option ----------
arrets = arrets[arrets["type"] != "trottinette_station"].reset_index(drop=True)   # zones virtuelles exclues
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()
arrets["nom_norm"] = arrets["nom"].map(norm)
arrets["x"], arrets["y"] = arrets.geometry.x, arrets.geometry.y
arrets["option_id"] = -1
suivant = 0
for _, idx in arrets.groupby(["ville", "type", "nom_norm"]).groups.items():
    idx = np.array(list(idx))
    if len(idx) == 1:
        lab = np.array([0])
    else:
        paires = cKDTree(arrets.loc[idx, ["x", "y"]].values).query_pairs(100, output_type="ndarray")
        mat = csr_matrix((np.ones(len(paires)), (paires[:, 0], paires[:, 1])), shape=(len(idx), len(idx)))
        _, lab = connected_components(mat, directed=False)
    arrets.loc[idx, "option_id"] = lab + suivant
    suivant += lab.max() + 1

options = arrets.groupby("option_id").agg(
    ville=("ville", "first"), type=("type", "first"), nom=("nom", "first"),
    departs_heure_pointe=("departs_heure_pointe", "sum"), capacite=("capacite", "sum"),
    nb_quais=("arret_id", "count"), x=("x", "mean"), y=("y", "mean")).reset_index()
print("Options après regroupement :")
print(options.pivot_table(index="type", columns="ville", values="option_id", aggfunc="count", fill_value=0))

# ---------- 2. Réseau piéton + distances réelles, ville par ville ----------
vers_l93 = Transformer.from_crs(4326, 2154, always_xy=True)
acces = []
for ville in ["toulon", "marseille"]:
    noeuds = pd.read_parquet(f"{BRONZE}/osm/{ville}/pieton_noeuds.parquet", columns=["id", "lon", "lat"]).drop_duplicates("id").reset_index(drop=True)
    tr = gpd.read_parquet(f"{BRONZE}/osm/{ville}/pieton_troncons.parquet")
    if "length" not in tr.columns:
        tr["length"] = tr.to_crs(2154).length
    pos = pd.Series(np.arange(len(noeuds)), index=noeuds["id"].astype("int64"))
    u = tr["u"].astype("int64").map(pos); v = tr["v"].astype("int64").map(pos); w = pd.to_numeric(tr["length"], errors="coerce")
    ok = u.notna() & v.notna() & w.notna()
    e = pd.DataFrame({"a": np.r_[u[ok], v[ok]].astype(int), "b": np.r_[v[ok], u[ok]].astype(int), "w": np.r_[w[ok], w[ok]]})
    e = e.groupby(["a", "b"], as_index=False)["w"].min()
    n = len(noeuds)
    G = csr_matrix((e.w.values, (e.a.values, e.b.values)), shape=(n, n))
    nx_, ny_ = vers_l93.transform(noeuds.lon.astype(float).values, noeuds.lat.astype(float).values)
    arbre = cKDTree(np.c_[nx_, ny_])

    c = carreaux[carreaux.ville == ville].reset_index(drop=True)
    d_c, n_c = arbre.query(np.c_[c.centroid.x, c.centroid.y])          # centre du carreau → rue la plus proche
    o = options[options.ville == ville].reset_index(drop=True)
    d_o, n_o = arbre.query(o[["x", "y"]].values)                        # arrêt → rue la plus proche
    o["noeud"], o["d_snap"] = n_o, d_o

    sources, morceaux = np.unique(n_o), []
    for i in range(0, len(sources), 150):
        src = sources[i:i + 150]
        D = dijkstra(G, directed=True, indices=src, limit=MARCHE)[:, n_c]
        r, k = np.nonzero(np.isfinite(D))
        morceaux.append(pd.DataFrame({"noeud": src[r], "i_carreau": k, "d_reseau": D[r, k]}))
    m = pd.concat(morceaux).merge(o[["option_id", "type", "noeud", "d_snap"]], on="noeud")
    m["distance_m"] = m.d_reseau + m.d_snap + d_c[m.i_carreau.values]
    m = m[m.distance_m <= MARCHE].copy()
    m["idcar_200m"] = c.idcar_200m.values[m.i_carreau.values]
    m["ville"] = ville
    m = m.sort_values("distance_m").drop_duplicates(["idcar_200m", "option_id"])
    acces.append(m[["ville", "idcar_200m", "option_id", "type", "distance_m"]])

    nb = m.groupby("idcar_200m").option_id.nunique()
    hab = c.set_index("idcar_200m")["ind"]
    print(f"{ville:10s} {len(m):>8,} liens carreau→option | habitants avec ≥1 option à 5 min : "
          f"{100 * hab[hab.index.isin(nb.index)].sum() / hab.sum():.1f} % | options moyennes par carreau : {nb.reindex(hab.index).fillna(0).mean():.1f}")

acces = pd.concat(acces, ignore_index=True)

# ---------- 3. Sauvegarder dans lakehouse_silver ----------
options_g = gpd.GeoDataFrame(options, geometry=gpd.points_from_xy(options.x, options.y), crs=2154)
options_g.to_parquet(f"{S}/acces_5min/options.parquet")
acces.to_parquet(f"{S}/acces_5min/acces_5min.parquet")

def vers_table(df, nom):
    t = pd.DataFrame(df.drop(columns="geometry")) if "geometry" in df else df.copy()
    if "geometry" in df:
        p = df.to_crs(4326).geometry
        t["lon"], t["lat"] = p.x, p.y
    for col in t.columns:
        if t[col].dtype == object: t[col] = t[col].astype(str)
    sdf = spark.createDataFrame(t)
    try:    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"lakehouse_silver.dbo.{nom}")
    except Exception: sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"lakehouse_silver.{nom}")
vers_table(options_g, "options")
vers_table(acces, "acces_5min")
print("Sauvegardé : tables 'options' et 'acces_5min' dans lakehouse_silver")

StatementMeta(, f45c4e44-fdc4-4b8a-9204-a08a4f607000, 11, Finished, Available, Finished, False)

Options après regroupement :
ville         marseille  toulon
type                           
bateau                2       8
bus                1315    1142
citiz                91       0
metro                30       0
telepherique          0       2
train                13       8
tram                 44       0
velo_station        228       0
toulon        4,822 liens carreau→option | habitants avec ≥1 option à 5 min : 76.5 % | options moyennes par carreau : 1.0
marseille     7,232 liens carreau→option | habitants avec ≥1 option à 5 min : 83.3 % | options moyennes par carreau : 2.3
Sauvegardé : tables 'options' et 'acces_5min' dans lakehouse_silver


In [5]:
print(options.pivot_table(index="type", columns="ville", values="option_id", aggfunc="count", fill_value=0))
print()
print(acces.groupby(["ville", "type"]).idcar_200m.nunique().unstack(0).fillna(0).astype(int).rename_axis("carreaux qui atteignent ce type"))

StatementMeta(, f45c4e44-fdc4-4b8a-9204-a08a4f607000, 12, Finished, Available, Finished, False)

ville         marseille  toulon
type                           
bateau                2       8
bus                1315    1142
citiz                91       0
metro                30       0
telepherique          0       2
train                13       8
tram                 44       0
velo_station        228       0

ville                            marseille  toulon
carreaux qui atteignent ce type                   
bateau                                   7      21
bus                                   1996    2189
citiz                                  252       0
metro                                   61       0
telepherique                             0       4
train                                   44      29
tram                                   143       0
velo_station                           701       0
